In [0]:
# Fahrlytics — Silver Customers Processing

# COMMAND ----------
# Configuration & City Normalization Map
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import time

CATALOG = "fahrlytics"
BRONZE  = f"{CATALOG}.bronze"
SILVER  = f"{CATALOG}.silver"

spark.sql(f"USE CATALOG {CATALOG}")

# Valid Canonical German Cities
VALID_CITIES = ["Berlin", "Munich", "Hamburg", "Frankfurt", "Cologne"]

# Map typical raw dirty variations generated in Bronze
GERMAN_CITY_MAP = {
    "berlin": "Berlin",
    "berlin-mitte": "Berlin",
    "berln": "Berlin",
    "münchen": "Munich",
    "munich": "Munich",
    "muenchen": "Munich",
    "münchen-stadt": "Munich",
    "hamburg": "Hamburg",
    "hh": "Hamburg",
    "hamburg-nord": "Hamburg",
    "frankfurt": "Frankfurt",
    "frankfurt am main": "Frankfurt",
    "ffm": "Frankfurt",
    "frankfurt/m": "Frankfurt",
    "köln": "Cologne",
    "koeln": "Cologne",
    "cologne": "Cologne"
}

def clean_city_expr(col_name):
    expr = F.when(F.lit(False), F.lit(None))
    for dirty, clean in GERMAN_CITY_MAP.items():
        expr = expr.when(F.lower(F.trim(F.col(col_name))) == dirty, F.lit(clean))
    return expr.otherwise(F.col(col_name))

print(f"Catalog : {CATALOG}")
print(f"Source  : {BRONZE}.customers")
print(f"Target  : {SILVER}.customers")



In [0]:
# Check Target State & Watermark
table_exists = spark.catalog.tableExists(f"{SILVER}.customers")
is_empty = not table_exists or spark.table(f"{SILVER}.customers").count() == 0

if is_empty:
    LOAD_TYPE = "full"
    df_bronze = spark.table(f"{BRONZE}.customers")
    print("Initiating FULL load for customers.")
else:
    LOAD_TYPE = "incremental"
    last_run = spark.sql(f"SELECT MAX(_processed_at) FROM {SILVER}.customers").collect()[0][0]
    df_bronze = spark.table(f"{BRONZE}.customers").filter(F.col("_ingested_at") > last_run)
    print(f"Initiating INCREMENTAL load for rows ingested after: {last_run}")

input_count = df_bronze.count()
print(f"Incoming rows: {input_count:,}")

if input_count == 0:
    dbutils.notebook.exit("No new customer records to process.")

In [0]:
# Deduplication & City Standardization
# Retain the latest signup/ingestion record per customer_id
window_spec = Window.partitionBy("customer_id").orderBy(
    F.col("_ingested_at").desc(), 
    F.col("signup_date").desc()
)

deduped_customers = (
    df_bronze
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .withColumn("city_raw", F.col("city"))
    .withColumn("city", clean_city_expr("city"))
)

In [0]:
# Rating Sanitization & Quality Flags
df_rated = (
    deduped_customers
    .withColumn("rating_raw", F.col("rating"))
    .withColumn("is_rating_invalid", (F.col("rating") > 5.0) | (F.col("rating") < 1.0))
    .withColumn(
        "rating_cleaned",
        F.when((F.col("rating") > 5.0) | (F.col("rating") < 1.0), None)
        .otherwise(F.col("rating"))
    )
)

# Compute city averages to impute nulls/invalid ratings
city_benchmarks = (
    df_rated.filter(F.col("rating_cleaned").isNotNull())
    .groupBy("city")
    .agg(F.round(F.avg("rating_cleaned"), 2).alias("avg_city_rating"))
)

df_clean = (
    df_rated
    .join(F.broadcast(city_benchmarks), on="city", how="left")
    .withColumn("rating", F.coalesce(F.col("rating_cleaned"), F.col("avg_city_rating"), F.lit(4.50)))
    .drop("avg_city_rating", "rating_cleaned")
    .withColumn(
        "is_churn_inconsistent",
        F.when((F.col("is_churned") == True) & F.col("churn_date").isNull(), True)
        .when((F.col("is_churned") == False) & F.col("churn_date").isNotNull(), True)
        .otherwise(False)
    )
    .withColumn("_processed_at", F.current_timestamp())
)

In [0]:
# Create Target Silver Table Schema (if not exists)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}.customers (
        customer_id             STRING,
        name                    STRING,
        city_raw                STRING,
        city                    STRING,
        signup_date             DATE,
        device_type             STRING,
        referral_source         STRING,
        referred_by_customer_id STRING,
        rating_raw              DOUBLE,
        rating                  DOUBLE,
        is_rating_invalid       BOOLEAN,
        is_churned              BOOLEAN,
        churn_date              DATE,
        is_churn_inconsistent   BOOLEAN,
        _ingested_at            TIMESTAMP,
        _processed_at           TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (city)
    COMMENT 'Silver conformed and deduplicated customer dimension'
""")

In [0]:
# Write / Upsert to Silver
t0 = time.time()

if LOAD_TYPE == "full":
    (
        df_clean.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("city")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER}.customers")
    )
    print(f"Full customer load written: {df_clean.count():,} rows")
else:
    target_table = DeltaTable.forName(spark, f"{SILVER}.customers")
    (
        target_table.alias("target")
        .merge(
            df_clean.alias("source"),
            "target.city = source.city AND target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Incremental customer merge finished.")

print(f"Write duration: {round(time.time() - t0, 2)}s")

# File compaction
spark.sql(f"OPTIMIZE {SILVER}.customers")
print("Delta OPTIMIZE completed.")

In [0]:
# Verification
df_verify = spark.table(f"{SILVER}.customers")
print(f"Total customers in silver: {df_verify.count():,}")
df_verify.groupBy("city").count().orderBy("count", ascending=False).show()